In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pathlib import Path
import hashlib, urllib.request, zipfile, subprocess, sys, importlib.util

WORK = Path("/content/drive/MyDrive/MiFO")
for sub in ["data/raw/fakenewsnet", "data/raw/liar", "data/processed/step3_pilot", "data/processed/step3_full"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

print("1. Downloading FakeNewsNet CSVs to Drive...")
BASE = "https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset"
for name in ["politifact_fake.csv", "politifact_real.csv", "gossipcop_fake.csv", "gossipcop_real.csv"]:
    dest = WORK / "data/raw/fakenewsnet" / name
    if not dest.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}", dest)
        print(f"   Downloaded {name}")
    else:
        print(f"   Already exists: {name}")

print("\n2. Downloading LIAR dataset to Drive...")
zpath = WORK / "data/raw/liar_dataset.zip"
if not (WORK / "data/raw/liar/train.tsv").exists():
    if not zpath.exists():
        try:
            urllib.request.urlretrieve("https://www.cs.ucsb.edu/~william/data/liar_dataset.zip", zpath)
        except Exception:
            # Fallback mirror if UCSB blocks
            urllib.request.urlretrieve("https://raw.githubusercontent.com/barun-saha/liar-plus/master/data/train.tsv", WORK / "data/raw/liar/train.tsv")
            urllib.request.urlretrieve("https://raw.githubusercontent.com/barun-saha/liar-plus/master/data/test.tsv", WORK / "data/raw/liar/test.tsv")
            urllib.request.urlretrieve("https://raw.githubusercontent.com/barun-saha/liar-plus/master/data/val.tsv", WORK / "data/raw/liar/valid.tsv")
    if zpath.exists() and not (WORK / "data/raw/liar/train.tsv").exists():
        with zipfile.ZipFile(zpath) as z:
            z.extractall(WORK / "data/raw/liar")
    print("   LIAR ready!")
else:
    print("   LIAR already exists!")

print("\n3. Installing dependencies...")
for pkg in ["sklearn", "trafilatura"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\nSUCCESS! WORK =", WORK)
print("Data is now permanently saved in your Google Drive.")

1. Downloading FakeNewsNet CSVs to Drive...
   Already exists: politifact_fake.csv
   Already exists: politifact_real.csv
   Already exists: gossipcop_fake.csv
   Already exists: gossipcop_real.csv

2. Downloading LIAR dataset to Drive...
   LIAR already exists!

3. Installing dependencies...

SUCCESS! WORK = /content/drive/MyDrive/MiFO
Data is now permanently saved in your Google Drive.


In [6]:
import importlib.util, subprocess, sys
from pathlib import Path

print("platform:", sys.platform, "| cwd:", Path.cwd())
print("'C:/Users/anush/MiFO' exists:", Path("C:/Users/anush/MiFO").exists())

WORK = None
for cand in [Path("C:/Users/anush/MiFO"), Path("/mnt/c/Users/anush/MiFO"),
             Path("/content/drive/MyDrive/MiFO")]:
    if (cand / "data/raw/fakenewsnet/politifact_fake.csv").exists():
        WORK = cand; break
assert WORK is not None, "Auto-locate failed — paste the lines above back to me."
print("WORK =", WORK)

for pkg in ["sklearn", "trafilatura"]:
    if importlib.util.find_spec(pkg) is None:
        print("installing", pkg)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
        if r.returncode != 0:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "--break-system-packages", pkg])
print("deps ready")

platform: linux | cwd: /content
'C:/Users/anush/MiFO' exists: False
WORK = /content/drive/MyDrive/MiFO
deps ready


In [7]:
#Cell A1
import pandas as pd, numpy as np
from urllib.parse import urlparse

RAW = WORK / "data/raw/fakenewsnet"
frames = []
for g in ["politifact", "gossipcop"]:
    for l in ["fake", "real"]:
        df = pd.read_csv(RAW / f"{g}_{l}.csv")
        df["source_group"], df["label_name"], df["label"] = g, l, (l == "fake")
        frames.append(df)
fn = pd.concat(frames, ignore_index=True)

def domain_of(u):
    u = str(u).strip()
    if not u or u.lower() == "nan": return ""
    if not u.startswith(("http://", "https://")): u = "http://" + u
    net = urlparse(u).netloc.lower()
    return net[4:] if net.startswith("www.") else net
fn["domain"] = fn["news_url"].map(domain_of)

# minority-mass closure (owed from step 2 — expect ~3,800, near the LOO errors)
d = fn[fn["domain"] != ""].groupby("domain")["label"].agg(n="size", n_fake="sum")
print(f"Minority-label articles across domains: "
      f"{int(np.minimum(d['n_fake'], d['n'] - d['n_fake']).sum())}\n")

d["bucket"] = np.where(d["n_fake"]/d["n"] >= 0.9, "clean_fake",
              np.where(d["n_fake"]/d["n"] <= 0.1, "clean_real", "MIXED"))
fn = fn.merge(d[["bucket"]], left_on="domain", right_index=True, how="left")
fn["bucket"] = fn["bucket"].fillna("no_domain")

pool = fn[(fn["domain"] != "") & (fn["domain"] != "web.archive.org")]
pool = pool.drop_duplicates(subset="news_url")

parts = [pool[pool.source_group == "politifact"]]
for (b, l), s in pool[pool.source_group == "gossipcop"].groupby(["bucket", "label_name"]):
    parts.append(s.sample(min(len(s), 125), random_state=42))
pilot = pd.concat(parts, ignore_index=True)

print(pilot.groupby(["source_group", "bucket", "label_name"]).size().to_string())
print("\nPilot URLs:", len(pilot))
out = WORK / "data/processed/step3_pilot"; out.mkdir(parents=True, exist_ok=True)
pilot.to_csv(out / "pilot_sample.csv", index=False)

Minority-label articles across domains: 3184

source_group  bucket      label_name
gossipcop     MIXED       fake          125
                          real          125
              clean_fake  fake          125
                          real           27
              clean_real  fake          125
                          real          125
politifact    MIXED       fake           57
                          real          117
              clean_fake  fake          290
                          real            1
              clean_real  fake           12
                          real          301

Pilot URLs: 1430


In [8]:
#Cell A2
import requests, time, threading, trafilatura, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

PILOT = pd.read_csv(WORK / "data/processed/step3_pilot/pilot_sample.csv")
TXT_DIR = WORK / "data/raw/crawl_pilot"; TXT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = WORK / "data/processed/step3_pilot/results.csv"

# EDIT: put your real email here — honest crawlers identify themselves
UA_RESEARCH = {"User-Agent": "MiFOResearchBot/0.1 (academic study; contact: YOU@EMAIL.COM)"}
UA_BROWSER = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

domain_last, lock = defaultdict(float), threading.Lock()
MIN_GAP = 2.0  # seconds between hits to the same domain

def polite_wait(dom):
    while True:
        with lock:
            if time.time() >= domain_last[dom] + MIN_GAP:
                domain_last[dom] = time.time(); return
        time.sleep(0.5)

def fetch(url):
    url = str(url).strip()
    if not url.startswith(("http://", "https://")): url = "http://" + url
    try:
        r = requests.get(url, headers=UA_RESEARCH, timeout=15, allow_redirects=True)
        if r.status_code in (403, 429, 503):   # one polite retry with browser UA
            time.sleep(3)
            r = requests.get(url, headers=UA_BROWSER, timeout=15, allow_redirects=True)
        return r
    except Exception as e:
        return None

def crawl_row(row):
    polite_wait(row["domain"])
    r = fetch(row["news_url"])
    rec = {"id": row["id"], "group": row["source_group"], "label": row["label_name"],
           "bucket": row["bucket"], "url": row["news_url"], "status": None,
           "final_url": None, "text_len": 0}
    if r is not None:
        rec["status"], rec["final_url"] = r.status_code, r.url
        if r.status_code == 200 and "html" in r.headers.get("Content-Type", "").lower():
            text = trafilatura.extract(r.text, include_comments=False,
                                       include_tables=False) or ""
            rec["text_len"] = len(text)
            if text:
                (TXT_DIR / f"{row['id']}.txt").write_text(text, encoding="utf-8")
    else:
        rec["status"] = "error"
    return rec

rows, t0 = [], time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    futs = [ex.submit(crawl_row, r) for r in PILOT.to_dict("records")]
    for i, f in enumerate(as_completed(futs), 1):
        rows.append(f.result())
        if i % 100 == 0:
            pd.DataFrame(rows).to_csv(RESULTS, index=False)
            print(f"{i}/{len(PILOT)} done ({time.time()-t0:.0f}s)")
pd.DataFrame(rows).to_csv(RESULTS, index=False)
print(f"FINISHED {len(rows)} in {(time.time()-t0)/60:.1f} min -> {RESULTS}")

100/1430 done (15s)


200/1430 done (34s)


300/1430 done (50s)


400/1430 done (64s)
500/1430 done (81s)
600/1430 done (101s)
700/1430 done (119s)


800/1430 done (142s)


...

1000/1430 done (178s)


1100/1430 done (207s)


1200/1430 done (243s)


ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None


1300/1430 done (267s)
1400/1430 done (283s)
FINISHED 1430 in 5.0 min -> /content/drive/MyDrive/MiFO/data/processed/step3_pilot/results.csv


In [9]:
#Cell C2
import pandas as pd, numpy as np
COLS = ["json_id","label","statement","subjects","speaker","job","state","party",
        "barely_true","false","half_true","mostly_true","pants_fire","context"]
liar = {s: pd.read_csv(WORK / f"data/raw/liar/{s}.tsv", sep="\t",
                       names=COLS, quoting=3) for s in ["train","valid","test"]}
tr = liar["train"].copy()
tr["words"] = tr["statement"].str.split().str.len()

print("=== Statement length (words) by class ===")
print(tr.groupby("label")["words"].agg(["count","median","mean"]).round(1)
      .sort_values("median").to_string())

print(f"\n=== Speakers: {tr['speaker'].nunique()} unique; top 5 ===")
print(tr["speaker"].value_counts().head(5).to_string())

print("\n=== Credit history (mean prior counts) by current class ===")
hist = ["barely_true","false","half_true","mostly_true","pants_fire"]
h = tr[hist].apply(pd.to_numeric, errors="coerce")
h["label"] = tr["label"].values
print(h.groupby("label").mean().round(1).to_string())

subj = tr["subjects"].fillna("").str.split(",").explode().str.strip()
print("\n=== Top subjects ===");  print(subj[subj != ""].value_counts().head(10).to_string())

=== Statement length (words) by class ===
             count  median  mean
label                           
false         1998    15.0  16.8
pants-fire     842    16.0  17.1
true          1683    16.0  17.9
barely-true   1657    17.0  18.1
mostly-true   1966    17.0  18.2
half-true     2123    18.0  18.8

=== Speakers: 2916 unique; top 5 ===
speaker
barack-obama       493
donald-trump       274
hillary-clinton    239
mitt-romney        180
scott-walker       150

=== Credit history (mean prior counts) by current class ===
             barely_true  false  half_true  mostly_true  pants_fire
label                                                              
barely-true         11.7   12.4       14.8         13.9         5.4
false               11.7   15.9       15.2         14.0         7.9
half-true           11.9   12.6       19.6         18.2         4.5
mostly-true         11.8   12.3       19.8         20.6         3.8
pants-fire          11.0   18.1       11.5          9.2        1

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score

pipe = make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
                     LogisticRegression(max_iter=2000))
pipe.fit(liar["train"]["statement"], liar["train"]["label"])
pred = pipe.predict(liar["test"]["statement"])

maj = liar["train"]["label"].value_counts(normalize=True).iloc[0]
acc = accuracy_score(liar["test"]["label"], pred)
f1m = f1_score(liar["test"]["label"], pred, average="macro")
print(f"majority baseline: {maj:.3f}  |  TF-IDF+LR accuracy: {acc:.3f}  |  macro-F1: {f1m:.3f}\n")
print(classification_report(liar["test"]["label"], pred, digits=2))

out = WORK / "data/processed/step3_liar"; out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"y": liar["test"]["label"], "pred": pred}).to_csv(out / "lr_baseline_preds.csv", index=False)
print("saved ->", out / "lr_baseline_preds.csv")

majority baseline: 0.207  |  TF-IDF+LR accuracy: 0.252  |  macro-F1: 0.219

              precision    recall  f1-score   support

 barely-true       0.23      0.14      0.18       214
       false       0.31      0.40      0.35       250
   half-true       0.22      0.29      0.25       267
 mostly-true       0.23      0.27      0.25       249
  pants-fire       0.33      0.03      0.06        92
        true       0.25      0.21      0.23       211

    accuracy                           0.25      1283
   macro avg       0.26      0.22      0.22      1283
weighted avg       0.25      0.25      0.24      1283

saved -> /content/drive/MyDrive/MiFO/data/processed/step3_liar/lr_baseline_preds.csv


In [11]:
#Cell A3
res = pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
res["live"] = res["status"] == 200
res["extracted"] = res["text_len"] >= 200

print(f"HTTP 200: {res['live'].mean():.1%}  |  extracted(>=200 chars): "
      f"{res['extracted'].mean():.1%}  |  median text len: "
      f"{res.loc[res.extracted, 'text_len'].median():.0f}")

print("\n=== By group/label ===")
print(res.groupby(["group","label"]).agg(n=("id","size"), live=("live","mean"),
      extracted=("extracted","mean")).round(3).to_string())

print("\n=== By bucket ===")
print(res.groupby("bucket").agg(n=("id","size"), live=("live","mean"),
      extracted=("extracted","mean")).round(3).to_string())

print("\n=== Status codes ===")
print(res["status"].value_counts(dropna=False).head(10).to_string())

HTTP 200: 0.0%  |  extracted(>=200 chars): 46.2%  |  median text len: 2498

=== By group/label ===
                    n  live  extracted
group      label                      
gossipcop  fake   375   0.0      0.627
           real   277   0.0      0.639
politifact fake   359   0.0      0.312
           real   419   0.0      0.325

=== By bucket ===
              n  live  extracted
bucket                          
MIXED       424   0.0      0.573
clean_fake  443   0.0      0.379
clean_real  563   0.0      0.442

=== Status codes ===
status
200      797
error    189
404      175
403      150
402       51
202       15
400       14
410       11
500        6
503        5


In [12]:
#extra A3
res["live"] = pd.to_numeric(res["status"], errors="coerce") == 200
print("Real HTTP 200 Live Rate:", res["live"].mean())

Real HTTP 200 Live Rate: 0.5573426573426573


In [13]:
import requests, pandas as pd, time

res = pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
failed = res[(pd.to_numeric(res["status"], errors="coerce") != 200)
             | (res["text_len"] < 200)].copy()
sample = failed.sample(min(20, len(failed)), random_state=42)

for i, (_, r) in enumerate(sample.iterrows(), 1):
    url = str(r["url"]).strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    try:
        resp = requests.get("https://archive.org/wayback/available",
                            params={"url": url}, timeout=20)
        print(f"{i:2d}  {resp.status_code}  {resp.text[:150]}")
    except Exception as e:
        print(f"{i:2d}  EXCEPTION {type(e).__name__}: {str(e)[:80]}")
    time.sleep(2)

 1  200  {"url": "http://www.sohh.com/teyana-taylor-tickled-by-kanye-west-dating-gossip-they-got-me-dien-over-here/", "archived_snapshots": {"closest": {"statu
 2  200  {"url": "http://beautyzon.info/spencer-pratts-guide-to-street-style-stardom-trap-music-fanny-packs-kim-kardashian/", "archived_snapshots": {}}
 3  200  {"url": "https://1043myfm.iheart.com/content/2017-06-16-jamie-foxx-ansel-elgort-james-corden-battle-in-epic-riff-off/", "archived_snapshots": {"closes
 4  200  {"url": "https://people.com/style/kim-kardashian-kylie-kendall-jenner-business-fashion-dinner/", "archived_snapshots": {"closest": {"status": "200", "
 5  200  {"url": "http://feedbox.com/2018/07/ariana-grande-wants-to-spread-the-f-king-light-in-this-chaotic-time/", "archived_snapshots": {}}
 6  200  {"url": "https://washingtonpress.com/2018/06/25/a-trump-fan-was-just-charged-with-trying-to-murder-maxine-waters-on-same-day-trump-threatened-her/", "
 7  200  {"url": "http://polls.trendolizer.com/2017/12/alabama-sta

In [14]:
import pandas as pd
from urllib.parse import urlparse

RAW = WORK / "data/raw/fakenewsnet"
frames = []
for g in ["politifact", "gossipcop"]:
    for l in ["fake", "real"]:
        df = pd.read_csv(RAW / f"{g}_{l}.csv")
        df["source_group"], df["label_name"], df["label"] = g, l, (l == "fake")
        frames.append(df)
fn = pd.concat(frames, ignore_index=True)

def domain_of(u):
    u = str(u).strip()
    if not u or u.lower() == "nan": return ""
    if not u.startswith(("http://", "https://")): u = "http://" + u
    net = urlparse(u).netloc.lower()
    return net[4:] if net.startswith("www.") else net
fn["domain"] = fn["news_url"].map(domain_of)

pool = fn[(fn["domain"] != "") & (fn["domain"] != "web.archive.org")]
pool = pool.drop_duplicates(subset="news_url")

pilot_done = set(pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
                 .query("text_len >= 200")["id"])
manifest = pool[~pool["id"].isin(pilot_done)].copy()

out = WORK / "data/processed/step3_full"; out.mkdir(parents=True, exist_ok=True)
manifest.to_csv(out / "manifest.csv", index=False)
print(f"unique URLs: {len(pool)} | already have text (pilot): {len(pilot_done)}")
print(f"TO CRAWL: {len(manifest)}")
print(manifest.groupby(["source_group", "label_name"]).size().to_string())

unique URLs: 21461 | already have text (pilot): 660
TO CRAWL: 20801
source_group  label_name
gossipcop     fake           4446
              real          15825
politifact    fake            247
              real            283


In [15]:
# F5.1 — Stable Batch-Fed Full Crawl (Crash-Proof)
import subprocess, time, threading, trafilatura, pandas as pd, zipfile, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from pathlib import Path

UA_EMAIL = "anushaanand1543@gmail.com"
WORK  = Path("/content/drive/MyDrive/MiFO")
MAN   = pd.read_csv(WORK / "data/processed/step3_full/manifest.csv")
RES_D = WORK / "data/processed/step3_full/results.csv"
RES_L = Path("/content/results_local.csv")
TXT_L = Path("/content/crawl_full"); TXT_L.mkdir(exist_ok=True)
TXT_D = WORK / "data/raw/crawl_full"
ZIP_D = WORK / "data/processed/step3_full/texts_snapshot.zip"

UA_R = f"MiFOResearchBot/0.1 (academic study; contact: {UA_EMAIL})"
UA_B = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"

# 1. Recover any existing texts from local & Drive
rescued = {p.stem: p.stat().st_size for p in TXT_L.glob("*.txt")}
if TXT_D.exists():
    for p in TXT_D.glob("*.txt"):
        dest = TXT_L / p.name
        if not dest.exists():
            dest.write_bytes(p.read_bytes())
            rescued[p.stem] = dest.stat().st_size
print(f"Texts already saved on local disk: {len(rescued)}")

# 2. Check completed IDs
frames = []
if RES_L.exists(): frames.append(pd.read_csv(RES_L))
if RES_D.exists(): frames.append(pd.read_csv(RES_D))
done_ids, rec_rows = set(), []
if frames:
    prev = pd.concat(frames, ignore_index=True).drop_duplicates(subset="id", keep="last")
    done_ids |= set(prev["id"]); rec_rows.append(prev)

back = MAN[MAN["id"].isin(set(rescued) - done_ids)]
if len(back):
    rec_rows.append(pd.DataFrame({
        "id": back["id"], "group": back["source_group"], "label": back["label_name"],
        "bucket": back.get("bucket", "unknown"), "url": back["news_url"], "status": 200,
        "final_url": None, "text_len": [rescued[i] for i in back["id"]]}))
    done_ids |= set(back["id"])
if rec_rows:
    pd.concat(rec_rows, ignore_index=True).to_csv(RES_L, index=False)

todo = MAN[~MAN["id"].isin(done_ids)].sample(frac=1, random_state=42)
print(f"Manifest: {len(MAN)} | Already Done: {len(done_ids)} | Remaining Todo: {len(todo)}")

# 3. cURL fetcher (10s hard max)
domain_last, lock = defaultdict(float), threading.Lock()
MIN_GAP = 1.0

def polite_wait(dom):
    while True:
        with lock:
            if time.time() >= domain_last[dom] + MIN_GAP:
                domain_last[dom] = time.time(); return
        time.sleep(0.3)

def _curl_get(url, ua, max_time=10):
    try:
        r = subprocess.run(
            ["curl", "-s", "-L", "--compressed", "--max-time", str(max_time),
             "-A", ua, "-w", "\n__MIFO__%{http_code}", url],
            capture_output=True, text=True, errors="replace", timeout=max_time + 3)
        if r.returncode != 0 or "__MIFO__" not in r.stdout:
            return None
        body, code = r.stdout.rsplit("\n__MIFO__", 1)
        class _R: pass
        resp = _R()
        resp.status_code = int(code) if code.isdigit() else 0
        resp.text = body
        resp.headers = {"Content-Type": "text/html" if body.lstrip()[:1] == "<" else "other"}
        return resp
    except Exception:
        return None

def fetch(url):
    url = str(url).strip()
    if not url.startswith(("http://", "https://")): url = "http://" + url
    r = _curl_get(url, UA_R)
    if r is not None and r.status_code in (403, 429, 503):
        time.sleep(2)
        r = _curl_get(url, UA_B)
    return r

def crawl_row(row):
    polite_wait(row.get("domain", ""))
    r = fetch(row["news_url"])
    rec = {"id": row["id"], "group": row["source_group"], "label": row["label_name"],
           "bucket": row.get("bucket", "unknown"), "url": row["news_url"], "status": None,
           "final_url": None, "text_len": 0}
    if r is not None:
        rec["status"], rec["final_url"] = r.status_code, row["news_url"]
        if r.status_code == 200 and "html" in r.headers.get("Content-Type", "").lower():
            try:
                text = trafilatura.extract(r.text, include_comments=False, include_tables=False) or ""
            except Exception:
                text = ""
            rec["text_len"] = len(text)
            if text:
                (TXT_L / f"{row['id']}.txt").write_text(text, encoding="utf-8")
    else:
        rec["status"] = "error"
    return rec

# 4. Heartbeat
stats = {"done": len(done_ids), "texts": len(rescued), "t0": time.time()}
stop = threading.Event()
def heartbeat():
    time.sleep(30)
    while not stop.is_set():
        el = time.time() - stats["t0"]
        print(f"  [hb {el/60:4.1f}m] progress={stats['done']}/{len(MAN)} "
              f"texts={stats['texts']} ({stats['done']/max(1, el):.1f}/s)", flush=True)
        time.sleep(60)
threading.Thread(target=heartbeat, daemon=True).start()

def sync_to_drive():
    if RES_L.exists():
        pd.read_csv(RES_L).to_csv(RES_D, index=False)
    zp = "/content/texts_snapshot.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in TXT_L.glob("*.txt"):
            z.write(p, arcname=p.name)
    shutil.copy(zp, ZIP_D)
    print(f"  [sync] Snapshot saved to Drive", flush=True)

# 5. Safe Batch Execution (12 workers, in chunks of 300)
BATCH_SIZE = 300
all_records = todo.to_dict("records")

for start_idx in range(0, len(all_records), BATCH_SIZE):
    batch = all_records[start_idx : start_idx + BATCH_SIZE]
    rows = []
    with ThreadPoolExecutor(max_workers=12) as ex:
        futs = [ex.submit(crawl_row, r) for r in batch]
        for f in as_completed(futs):
            rec = f.result()
            rows.append(rec)
            stats["done"] += 1
            if rec["text_len"]: stats["texts"] += 1
    
    # Save after each batch
    done = pd.concat([pd.read_csv(RES_L), pd.DataFrame(rows)], ignore_index=True) if RES_L.exists() else pd.DataFrame(rows)
    done.drop_duplicates(subset="id", keep="last").to_csv(RES_L, index=False)
    
    if stats["done"] % 1500 < BATCH_SIZE:
        sync_to_drive()

stop.set()
sync_to_drive()

# 6. Final Status
res = pd.read_csv(RES_L if RES_L.exists() else RES_D)
res["ok"] = pd.to_numeric(res["status"], errors="coerce") == 200
print("\nFINISHED CRAWL!")
print(res.groupby(["group", "label"]).agg(n=("id", "size"), live=("ok", "mean"),
      with_text=("text_len", lambda s: (s >= 200).mean())).round(3).to_string())
print("\nTOTALS:", len(res), "attempted |", int((res["text_len"] >= 200).sum()), "with text")

  [hb 16.5m] progress=19996/20801 texts=3730 (20.2/s)


  [hb 17.5m] progress=20386/20801 texts=3944 (19.4/s)


  [hb 18.5m] progress=20742/20801 texts=4153 (18.7/s)
  [sync] Snapshot saved to Drive

FINISHED CRAWL!
                      n   live  with_text
group      label                         
gossipcop  fake    4446  0.604      0.567
           real   15825  0.625      0.602
politifact fake     247  0.105      0.008
           real     286  0.353      0.024

TOTALS: 20804 attempted | 12056 with text
